## Densità polifonica

Questo notebook carica una composizione e mostra quante voci sono attive in ogni momento.

### Importare il codice

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from crim_intervals import importScore

print('Ok')

Ok


### Caricamento della composizione

Caricamento di un file MusicXML dalla cartella locale `Music_Files/`. Modificare il valore di `filename` per cambiare composizione:

- `'Gabrieli_in_ecclesiis.xml'` — *In ecclesiis*, Giovanni Gabrieli
- `'Marenzio_o_voi.xml'` — *O voi che sospirate*, Luca Marenzio
- `'Monteverdi_o_rossignuol.xml'` — *O rossignuol*, Claudio Monteverdi
- `'Monteverdi_cantai.xml'` — *Cantai un tempo*, Claudio Monteverdi

Se il caricamento è andato a buon fine, `print(piece.metadata)` restituisce titolo e compositore.

In [ ]:
filename = 'Monteverdi_o_rossignuol.xml'  # ← cambia qui per caricare un altro file
url   = f'Music_Files/{filename}'
piece = importScore(url)
print(piece.metadata)

### Numero di voci attive

`piece.soundingCount()` restituisce, per ogni evento, il numero di voci di volta in volta attive (*sounding*).

In [ ]:
sound = piece.soundingCount()
sound

### Grafico

Lo stesso dato visualizzato come grafico a barre. Ogni colonna corrisponde a un evento; l'altezza indica il numero di voci attive.

In [ ]:
sound.plot.bar(figsize=(20, 4), width=1.0, xticks=[])
print('doppio click sul grafico per ingrandire/rimpicciolire')

### Distribuzione della densità polifonica

Il codice seguente restituisce una tabella che indica, per ciascun numero di voci attive (righe), la percentuale di durata rispetto alla durata totale della composizione.

In [ ]:
d = piece.durations()
total_duration = (d.index + d.max(axis=1)).max()

offsets   = sound.index.to_numpy()
durations = np.append(np.diff(offsets), total_duration - offsets[-1])

n_voices  = sound.max()
dist_pct  = pd.Series(
    index=range(n_voices + 1),
    data=[durations[sound.values == k].sum() / total_duration * 100
          for k in range(n_voices + 1)],
    name='percent',
)
dist_pct.index.name = 'sounding'

dist_pct.round(2).to_frame()

In [ ]:
ax = dist_pct.plot.barh(figsize=(7, 0.5 * (n_voices + 2)))
ax.set_xlabel('% della durata totale')
ax.set_ylabel('voci attive')
ax.set_title(f"{piece.metadata['title']} — {piece.metadata['composer']}")
ax.invert_yaxis()
for bar, val in zip(ax.patches, dist_pct):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)

---

## Confronto

Le celle seguenti calcolano la distribuzione della densità polifonica di più composizioni e restituiscono tabelle e grafici comparativi.

### Lista dei file da analizzare

Nelle due celle qui sotto ci sono **due versioni della lista**: la prima carica automaticamente tutti i file XML e MusicXML presenti nella cartella `Music_Files/` tramite `glob`; la seconda è una lista personalizzata che include soltanto le composizioni a 5 voci.

**Eseguire una sola delle due celle** prima di procedere con l'analisi.

---

**Come funziona `glob`**

`Path('Music_Files').glob('*.xml')` cerca nella cartella `Music_Files/` tutti i file il cui nome termina con `.xml`. Il carattere `*` è un *wildcard* che sta per "qualsiasi sequenza di caratteri". Vengono quindi raccolti anche i file `.musicxml`, `.mxl` e `.mei` con chiamate successive. Il risultato è una lista di oggetti `Path`, ciascuno dei quali rappresenta il percorso di un file.

In [ ]:
# OPZIONE A — tutti i file XML o MusicXML nella cartella Music_Files/
piece_list = sorted(Path('Music_Files').glob('*.xml')) + \
             sorted(Path('Music_Files').glob('*.musicxml'))

for p in piece_list:
    print(p)

In [ ]:
# OPZIONE B — solo brani a 5 voci
piece_list = [
    'Music_Files/Marenzio_o_voi.xml',          # Luca Marenzio
    'Music_Files/Monteverdi_o_rossignuol.xml', # Claudio Monteverdi
    'Music_Files/Monteverdi_cantai.musicxml',  # Claudio Monteverdi
]

for p in piece_list:
    print(p)

È possibile aggiungere altre liste personalizzate. Sarà sufficiente copiare la cella precedente e modificare i nomi dei files. Non c'è limite al numero di files per ciascuna lista.

In [ ]:
# LISTA PERSONALIZZATA (per esempio, soltanto Marenzio o voi e Monteverdi o rossignuol)
piece_list = [
    'Music_Files/Marenzio_o_voi.xml',          # modificare
    'Music_Files/Monteverdi_o_rossignuol.xml', # modificare
]

for p in piece_list:
    print(p)

Madrigali di Luca Marenzio dalla **[Marenzio Online Digital Edition (MODE)](http://www.marenzio.org):**

- Luca Marenzio, *Il quarto libro de' madrigali a sei voci*, a cura di Lucia Marchi,
  [MODE – Marenzio Online Digital Edition, 2024](https://marenzio.org/digital-edition/M-04-6.xhtml)
  — `M_04_6_01` – `M_04_6_15` (15 madrigali)

- Luca Marenzio, *Il sesto libro de madrigali a cinque voci*, a cura di Daniele Filippi,
  [MODE – Marenzio Online Digital Edition, 2019](https://marenzio.org/digital-edition/M-06-5.xhtml)
  — `M_06_5_01` – `M_06_5_17` (17 madrigali)

I file MEI sono stati scaricati da [github.com/marenzio/marenzio.github.io](https://github.com/marenzio/marenzio.github.io)
e sottoposti a una minima preparazione tecnica per la compatibilità con `crim-intervals`.
Per i dettagli vedere la sezione
[*Marenzio Corpus: Sources and Credits*](https://github.com/gtaschetti/analisi_mus_unipd#marenzio-corpus-sources-and-credits)
del README di questo repository.

In [ ]:
# Marenzio, Quarto libro a 6 vv e Sesto libro a 5
piece_list = sorted(str(p) for p in Path('Music_Files').glob('M_*.mei'))
piece_list

In [ ]:
# Marenzio, Quarto libro a 6 vv.
piece_list = sorted(str(p) for p in Path('Music_Files').glob('M_04_6*.mei'))
piece_list

In [ ]:
# Marenzio, Sesto libro a 5 vv.
piece_list = sorted(str(p) for p in Path('Music_Files').glob('M_06_5*.mei'))
piece_list

In [ ]:
# OPZIONE C - Madrigali della MODE - Marenzio Online Digital Edition

In [ ]:
def sounding_distribution(piece):
    """Return a time-weighted % distribution of sounding voice counts for a piece."""
    sound = piece.soundingCount()
    d     = piece.durations()
    total = (d.index + d.max(axis=1)).max()

    offsets   = sound.index.to_numpy()
    durations = np.append(np.diff(offsets), total - offsets[-1])
    n_voices  = sound.max()

    dist = pd.Series(
        index=range(n_voices + 1),
        data=[durations[sound.values == k].sum() / total * 100
              for k in range(n_voices + 1)],
        name='percent',
    )
    dist.index.name = 'sounding'
    return dist


results = {}   # label -> dist Series

for filepath in piece_list:
    piece_i = importScore(str(filepath))
    meta    = piece_i.metadata
    label   = f"{meta['composer']}  —  {meta['title']}"
    dist_i  = sounding_distribution(piece_i)
    results[label] = dist_i

    n_i = dist_i.index.max()
    ax  = dist_i.plot.barh(figsize=(7, 0.5 * (n_i + 2)))
    ax.set_xlabel('% della durata totale')
    ax.set_ylabel('voci attive')
    ax.set_title(label)
    ax.invert_yaxis()
    for bar, val in zip(ax.patches, dist_i):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

### Tabella comparativa

Ogni riga corrisponde a una composizione, ogni colonna corrisponde a un numero di voci attive. Ogni cella include la percentuale della durata della composizione corrispondente (riga) in cui è attivo un detarminato numero di voci (colonna); le celle vuote (0.0) indicano che quel numero di voci attive non è rappresentato nella composizione.

In [ ]:
max_voices = max(s.index.max() for s in results.values())
comparison = (pd.DataFrame(results)
                .T
                .reindex(columns=range(max_voices + 1))
                .fillna(0)
                .round(1))
comparison.index.name  = 'piece'
comparison.columns.name = 'sounding'

comparison

### Heatmap comparativa

Colore = % di durata. Un colore più intenso corrisponde a una maggiore durata (in percentuale) di una sonorità. Le colonne da sinistra a destra vanno dal mininimo al massimo numero di voci attive. Ogni composizione si legge come una riga: le celle più scure rappresentano la densità polifonica predominante.

In [ ]:
n_pieces = len(comparison)
fig, ax  = plt.subplots(figsize=(max(8, max_voices + 2), max(3, n_pieces * 0.7 + 1)))

im = ax.imshow(comparison.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='% della durata totale', shrink=0.8)

ax.set_xticks(range(max_voices + 1))
ax.set_xticklabels(range(max_voices + 1))
ax.set_xlabel('voci attive')
ax.set_yticks(range(n_pieces))
ax.set_yticklabels(comparison.index, fontsize=8)

for i in range(n_pieces):
    for j in range(max_voices + 1):
        val = comparison.iloc[i, j]
        if val > 0:
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=7,
                    color='white' if val > 55 else 'black')

print('doppio click per ingrandire/rimpicciolire il grafico')
plt.tight_layout()
plt.show()